In [ ]:
#meta for myFiftyoneComputerVision 6/2/2025. On-Prem. POC AI Doc Vision. Part 1. Full MLP -> Vis51 Data w/ Dimensionality Reduction
# refer to https://docs.voxel51.com/tutorials/dimension_reduction.html?highlight=umap
# code src: https://github.com/voxel51/fiftyone/blob/v1.4.0/docs/source/tutorials/dimension_reduction.ipynb


#infra: WLaptop + VSCode
#      env: default
#      confirmed Python 3.10.4
#      numpy 2.1.3, pandas 2.2.3, scikit-learn 1.6.1, matplotlib 3.10.0
#      pip 22.0.4, ipykernel 6.29.0, ipython 8.20.0
#fiftyone 1.4.0, fiftyone-brain 0.20.1, fiftyone_db 1.1.7
#umap-learn 0.5.7
#glob2 0.7


#input: 'IMAGES_SAMPLE-INVOICES-250' 
#       'DOC_DATA/df_sample-invoices-250_metadata.parquet'
#output: 'persist51_doc-sample-invoices-250'
#output: 51vis


#history
#6/2/2025 CREATE 51dataset (250 COUNT) FIRST TIME - DRAFT
#      Create and persist 1st time, so next time can just load
#      2 models x 3 dim reductions = 6 51vis
#      Sample dataset: Mendeley Data, Samples of electronic invoices (from 999 reduced to 250 for speedier POC)
#$note: No metadata to add field(s) to a dataset - better with with gt_label and a primitive field 
#$note: Need modifications to run the code after 51dataset is created and don't want to overwrite good things

#6/3/2025 FULL MLP: FROM IMAGES TO 51VIS (SAMPLE INVOICES 250) 
#      Use previously created metadata file
#      Create 51Dataset first time and Load exported
#      Load persisted 51dataset from disc

#6/9/2025 MISC CLEANUP (SAMPLE INVOICES 250) 
#      Rename files and folders
#      was DOC-SAMPLE-INVOICES-999_PDFs, DOC-SAMPLE-INVOICES-250_Images, df_sample_invoices_250_metadata.parquet
#      now PDFs_SAMPLE-INVOICES-250, IMAGES_SAMPLE-INVOICES-250, df_sample-invoices-250_metadata.parquet
# did not run


#$config  $error $fix

In [2]:
import sys
#import time as time
from glob import glob
import os

import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 150)

In [3]:
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.zoo as foz

In [ ]:
#----- GlOBAL VARS -----

#data #$config
DATA_PATH = 'DOC_DATA'
IMAGES_PATH =  'IMAGES_SAMPLE-INVOICES-250'
METADATA_SAMPLES_IN = DATA_PATH + '/df_sample-invoices-250_metadata.parquet' 

#dataset
DATASET_NAME = "doc-sample-invoices-250"

EXPORT_PATH='persist51_doc-sample-invoices-250'


# POC AI-DOC-VISION
## Visualizing Data with Dimensionality Reduction Techniques 

Using 51 walkthrough, run dimensionality reduction techniques (PCA, t-SNE, UMAP) on shipping docs in FiftyOne!

It covers the following:

- Why dimensionality reduction?
- Strengths and weaknesses of different dimensionality reduction techniques
- Running built-in dimensionality reduction techniques in FiftyOne

Embeddings — numeric vectors that represent features of your input data. In computer vision for instance, image embeddings are used in reverse image search applications. And in the context of large language models (LLMs), documents are chunked and embedded (with text embedding models) for retrieval augmented generation (RAG).

Embeddings are incredibly powerful, but given their high dimensionality (with lengths typically between 384 and 4096), they can be hard for humans to interpret and inspect. This is where dimensionality reduction techniques come in handy!

Dimensionality reduction techniques are quantitative methods for representing information from a higher dimensional space in a lower dimensional space. By squeezing our embeddings into two or three dimensions, we can visualize them to get a more intuitive understanding of the “hidden” structure in our data.

When we project high dimensional data into a low dimensional space, we implicitly make a trade-off between representational complexity and interpretability. To compress embeddings, dimensionality reduction techniques make assumptions about the underlying data, its distribution, and the relationships between variables.

3 popular dimensionality reduction techniques: PCA, t-SNE, and UMAP. We will give a brief overview of the strengths, weaknesses, and assumptions of each technique. And we will illustrate that both the model used to generate embeddings, and the dimensionality reduction technique play essential roles in shaping the visualization of your data.

*$note: dimensionality reduction techniques often have hyperparameters, which can have non-negligible impacts on the results. Here, we use the default hyperparameters (modify as you see fit!)*

## Setup

Using the FiftyOne library for data management and visualization  
- use [scikit-learn](https://scikit-learn.org/stable/) for PCA and t-SNE, and  
- [umap-learn](https://umap-learn.readthedocs.io/en/latest/#) for UMAP dimension reduction implementations:

In [5]:
#!pip install -U fiftyone scikit-learn umap-learn

## 0. Load Data
250 samples invoices

### 0a. Load tidy sample with metadata
previously created in `0_data` step

In [6]:
df_samples = pd.read_parquet(METADATA_SAMPLES_IN)
print(df_samples.shape)
print(df_samples.info())
df_samples.head()

(250, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   image_file_path  250 non-null    object
 1   image_file_name  250 non-null    object
dtypes: object(2)
memory usage: 4.0+ KB
None


,image_file_path,image_file_name
0,DOC-SAMPLE-INVOICES-250_Images/invoice_0_PAGE_0.jpg,invoice_0_PAGE_0.jpg
1,DOC-SAMPLE-INVOICES-250_Images/invoice_100_PAGE_0.jpg,invoice_100_PAGE_0.jpg
2,DOC-SAMPLE-INVOICES-250_Images/invoice_101_PAGE_0.jpg,invoice_101_PAGE_0.jpg
3,DOC-SAMPLE-INVOICES-250_Images/invoice_102_PAGE_0.jpg,invoice_102_PAGE_0.jpg
4,DOC-SAMPLE-INVOICES-250_Images/invoice_103_PAGE_0.jpg,invoice_103_PAGE_0.jpg


## 1. Create 51Dataset
Load sample images, include a gt_label + field(s)

In [7]:
# DATASET_NAME = "NAME TO DELETE" 

# #RUN ONLY IF WANT TO DELETE A DATASET
# #Snippet: Recreate dataset every time
# if DATASET_NAME in fo.list_datasets():
#     # Load your FiftyOne dataset
#     ds_docs = fo.load_dataset(DATASET_NAME)
#     ds_docs.delete() #$config
# else:
#     print("Create dataset")

# print(fo.list_datasets())

In [8]:
# #Snippet: Load the dataset from disc
# try:
#     # Load your FiftyOne dataset
#     dataset = fo.load_dataset(DATASET_NAME) #class 'fiftyone.core.dataset.Dataset'
# except ValueError:
# # If the dataset doesn't exist, create it from a directory of images
#     dataset = fo.Dataset.from_dir(DATA_PATH, dataset_type=fo.types.FiftyOneDataset, name=DATASET_NAME) #class 'fiftyone.core.dataset.Dataset'
#     dataset = fo.load_dataset(DATASET_NAME)
 
# print(fo.list_datasets())

In [9]:
# step: Load dataset 1st time - 250 docs
try:
    # Load your FiftyOne dataset
    dataset = fo.load_dataset(DATASET_NAME) #class 'fiftyone.core.dataset.Dataset'
except ValueError:
    # If the dataset doesn't exist, create it from a dir of images
    dataset = fo.Dataset.from_images_dir(IMAGES_PATH, name=DATASET_NAME) #class 'fiftyone.core.dataset.Dataset'
 
print(fo.list_datasets())

 100% |█████████████████| 250/250 [44.5ms elapsed, 0s remaining, 5.6K samples/s]   
['clustering-demo', 'clustering-demo2', 'doc-images', 'doc-images-2359', 'doc-sample-invoices-250']


In [10]:
# step: Load the full dataset 1st time
#list all images in dir of images
all_images = glob(IMAGES_PATH+"/*.jpg")
print(len(all_images))

all_images

250


['DOC-SAMPLE-INVOICES-250_Images\\invoice_0_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_100_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_101_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_102_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_103_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_104_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_105_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_106_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_107_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_108_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_109_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_10_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_110_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_111_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_112_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_113_PAGE_0.jpg',
 'DOC-SAMPLE-INVOICES-250_Images\\invoice_114_PAGE_0.jpg',


howto: Adding fields to a dataset  
refer to https://docs.voxel51.com/user_guide/using_datasets.html#adding-fields-to-a-sample  
https://docs.voxel51.com/user_guide/using_datasets.html#adding-fields-to-a-dataset

In [11]:
# # step: Load the full dataset 1st time $config - only if have ground truth and a primitive field
# #add gt_label and a field to FiftyOne dataset
# i=0
# for image in all_images:
#     #print(image)
#     sample = fo.Sample(
#         filepath=image,
#         ground_truth = fo.Classification(label=df_samples.iloc[i]['DOCUMENT_TYPE_NAME']))
#     dataset.add_sample(sample)
#     sample["carrier"] = df_samples.iloc[i]['CARRIER']
#     sample.save()

#     i += 1

# #ensure right media type
# dataset.media_type

In [12]:
dataset

Name:        doc-sample-invoices-250
Media type:  image
Num samples: 250
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField

In [13]:
dataset.stats(include_media=True)

Computing metadata...
 100% |█████████████████| 250/250 [205.3ms elapsed, 0s remaining, 1.2K samples/s]     


{'samples_count': 250,
 'samples_bytes': 65719,
 'samples_size': '64.2KB',
 'media_bytes': 50294973,
 'media_size': '48.0MB',
 'total_bytes': 50360692,
 'total_size': '48.0MB'}

In [14]:
# #preview
# dataset.count_values("ground_truth.label")

In [15]:
# #preview
# dataset.count_values("carrier")

In [16]:
#Launch if want to view 51dataset, images only
#session = fo.launch_app([ds_name])

### 1.1 Dimensionality reduction techniques
Compare and contrast our 3 dimensionality reduction techniques with two image embedding models: [ResNet-101](https://pytorch.org/vision/main/models/generated/torchvision.models.resnet101.html) and [CLIP](https://github.com/openai/CLIP). Whereas ResNet-101 is a more traditional vision model, representing the relationships between pixels and patches in images, CLIP captures more of the semantic content of the images.

We can load both from the [FiftyOne Model Zoo](https://docs.voxel51.com/user_guide/model_zoo/index.html):

In [17]:
clip = foz.load_zoo_model("clip-vit-base32-torch")
resnet101 = foz.load_zoo_model("resnet101-imagenet-torch")

Then generating embeddings for each model amounts to making a single call to the dataset’s `compute_embeddings()` method:

In [18]:
## compute and store resnet101 embeddings $note: time consuming
dataset.compute_embeddings(
    resnet101, 
    embeddings_field="resnet101_embeddings"
)

## compute and store clip embeddings 
dataset.compute_embeddings(
    clip, 
    embeddings_field="clip_embeddings"
)

 100% |█████████████████| 250/250 [11.8m elapsed, 0s remaining, 0.4 samples/s]    
 100% |█████████████████| 250/250 [17.1s elapsed, 0s remaining, 16.0 samples/s]      


### Tutorial: Dimensionality Reduction API in FiftyOne

Recap the API for running dimensionality reduction in FiftyOne.  
The [FiftyOne Brain](https://docs.voxel51.com/user_guide/brain.html) provides a [compute_visualization()](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_visualization) function that can be used to run dimensionality reduction on your data.

The first and only positional argument to this function is a sample collection, which can be either a [Dataset](https://docs.voxel51.com/user_guide/using_datasets.html#datasets) or a [DatasetView](https://docs.voxel51.com/user_guide/using_views.html).

Beyond that, you need to specify the following three things:
1. *What* you want to reduce the dimensionality of.
2. *How* you want to reduce the dimensionality.
3. *Where* you want to store the results.

### Tutorial 1. What to reduce the dimensionality of

There are multiple ways to specify what you would like dimension-reduced. Here are a few options:

- You can specify the name of the field containing the embeddings you would like to reduce using the `embeddings` argument. If your embeddings are stored in field "my_embeddings_field" on your samples, you would employ the syntax `embeddings="my_embeddings_field"`. This is useful if you need to reuse the same embeddings for multiple dimensionality reduction techniques, or for other brain methods.
- You can pass the embeddings in directly using as numpy array, also via the `embeddings` argument. This is useful if you have already computed your embeddings, and don’t need to store them on your samples.
- You can specify the *model* you would like to use to generate embeddings. This can be:
    - A `FiftyOne.Model` instance
    - The name (a string) of a model from the model zoo, in which case the model by that will be loaded from the FiftyOne Model Zoo.
    - A Hugging Face Transformers model, in which case the model will be converted to a `FiftyOne.Model` instance. See the [Hugging Face integration docs](https://docs.voxel51.com/integrations/huggingface.html) for more details.

### Tutorial 2. How to reduce the dimensionality

You can specify the base dimensionality reduction technique to use via the `method` argument. This can be one of the following strings: `pca`, `tsne`, `umap`, or `manual`.

For `pca`, `tsne`, and `umap`, you can specify the number of dimensions to reduce to via the `num_dims` argument. Additionally, you can specify hyperparameters for each technique as kwargs. For a complete description of available options, check out the visualization configs: 
[TSNEVisualizationConfig](https://docs.voxel51.com/api/fiftyone.brain.visualization.html#fiftyone.brain.visualization.TSNEVisualizationConfig),
[UMAPVisualizationConfig](https://docs.voxel51.com/api/fiftyone.brain.visualization.html#fiftyone.brain.visualization.UMAPVisualizationConfig), and [PCAVisualizationConfig](https://docs.voxel51.com/api/fiftyone.brain.visualization.html#fiftyone.brain.visualization.PCAVisualizationConfig).

You can use `method="manual"` if you already have the dimensionality-reduced data, and just want to store it on your samples for visualization purposes.

### Tutorial 3. Where to store the results

This is done via the `brain_key` argument. Once you have run the `compute_visualization()` method, you will be able to select this brain key in the FiftyOne App to visualize the results. You can also use the brain key to access the results programmatically:

In [19]:
# #Snippet: 
# import fiftyone as fo
# import fiftyone.brain as fob
# import fiftyone.zoo as foz

# #dataset = foz.load_zoo_dataset("cifar10", split="test")

# ## Compute PCA visualization
# fob.compute_visualization(
#     dataset,
#     embeddings="resnet101",
#     method="pca",
#     brain_key="resnet101_pca"
# )

# ## Access results
# pca_resnet_results = dataset_docs.load_brain_results("pca_resnet101")

#### 1.1a Dimensionality Reduction with PCA

[Principal Component Analysis](https://en.wikipedia.org/wiki/Principal_component_analysis), or PCA, is a dimensionality reduction technique that seeks to preserve as much variance as possible. Intuitively, PCA finds a set of orthogonal axes (principal components) that jointly “explain” as much of the variation in the data as possible. Mathematically, you can interpret PCA algorithms as effectively performing singular value decompositions and truncating the number of dimensions by eliminating the singular vectors with the smallest eigenvalues.

**Strengths**

- Simple, intuitive, and efficient for large datasets! 
- PCA is amenable to new data: If you have precomputed the transformation on an initial set of embeddings, you can apply that transformation to new embeddings and immediately visualize them in the same space.

**Limitations**

- Assumes that the relationships between variables are linear — an assumption which often does not hold when the inputs are embeddings, which themselves come from highly nonlinear deep neural networks
- Very susceptible to outliers.

#### 1.1a Running PCA on Embeddings

PCA is natively supported by the FiftyOne Brain’s `compute_visualization()`. To reduce dimensionality for a set of embeddings, we can specify the field the embeddings are stored in, and pass in `method="pca"`. In the app, we can open up an Embeddings panel to view the results:

In [20]:
## PCA with ResNet101 embeddings
fob.compute_visualization(
    dataset, 
    embeddings="resnet101_embeddings", 
    method="pca", 
    brain_key="resnet101_pca"
)

## PCA with CLIP embeddings
fob.compute_visualization(
    dataset, 
    embeddings="clip_embeddings", 
    method="pca", 
    brain_key="clip_pca"
)


Generating visualization...
Generating visualization...


In [21]:
## Access results
pca_resnet_results = dataset.load_brain_results("resnet101_pca")

### Tutorial: Vis

We can color by any attribute on our samples — in this case the ground truth label — and filter the contents of the sample grid interactively by selecting regions in the embeddings panel.

View and interpret both the CLIP and ResNet-101 embeddings:  
i.e.  
- the PCA plot does seem to very loosely retain information from the embeddings (and the original images). However, when we color by label, there is substantial overlap from one class to another. 
- Restricting the CLIP PCA view to just automobiles, trucks, and ships, we can see that the distributions for all three classes are essentially identical, aside from the ships extending slightly farther out.

#### 1.1b Dimensionality Reduction with t-SNE

[t-Distributed Stochastic Neighbor Embedding](https://en.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding), or t-SNE, is a nonlinear dimensionality reduction technique that aims to, roughly speaking, keep neighbors close. More precisely, t-SNE takes the initial, high-dimensional data (in our case embedding vectors) and computes the similarity between inputs. The algorithm then attempts to learn a lower-dimensional representation which preserves as much of the similarity as possible. Mathematically, this learning is achieved by minimizing the [Kullback-Leibler divergence](https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence) between the high-dimensional (fixed) and low-dimensional (trained) distributions.

**Strengths**

- t-SNE is nonlinear, making it a much better fit for (embeddings computed on) datasets like MNIST and CIFAR-10. 
- The technique is good at preserving local structure, making it easy to see clustering in data!

**Limitations**

- t-SNE relies on random initialization, so good fits are not guaranteed
Still sensitive to outliers
- Not scalable: for a dataset with n samples, t-SNE takes $\mathcal{O}(n^2)$ time to run, and requires $\mathcal{O}(n^2)$ space to operate


#### 1.1b Running t-SNE on Embeddings

t-SNE is natively supported by the FiftyOne Brain’s `compute_visualization()`, so we can run dimensionality reduction on our embeddings by passing `method="tsne"`:

In [22]:
#$error ValueError: n_components=50 must be between 1 and min(n_samples, n_features)=21 with svd_solver='randomized'

## t-SNE with ResNet101 embeddings
fob.compute_visualization(
    dataset, 
    embeddings="resnet101_embeddings", 
    method="tsne", 
    brain_key="resnet101_tsne"
)

## t-SNE with CLIP embeddings
fob.compute_visualization(
    dataset, 
    embeddings="clip_embeddings", 
    method="tsne", 
    brain_key="clip_tsne"
)

Generating visualization...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 250 samples in 0.000s...


c:\Users\chq-anyac\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[t-SNE] Computed neighbors for 250 samples in 0.742s...
[t-SNE] Computed conditional probabilities for sample 250 / 250
[t-SNE] Mean sigma: 0.251554
[t-SNE] Computed conditional probabilities in 0.012s
[t-SNE] Iteration 50: error = 52.9145889, gradient norm = 0.2802366 (50 iterations in 0.019s)
[t-SNE] Iteration 100: error = 51.0089722, gradient norm = 0.2631231 (50 iterations in 0.017s)
[t-SNE] Iteration 150: error = 49.2422485, gradient norm = 0.2973231 (50 iterations in 0.018s)
[t-SNE] Iteration 200: error = 48.8921852, gradient norm = 0.2902701 (50 iterations in 0.018s)
[t-SNE] Iteration 250: error = 48.6509895, gradient norm = 0.2920835 (50 iterations in 0.015s)
[t-SNE] KL divergence after 250 iterations with early exaggeration: 48.650990
[t-SNE] Iteration 300: error = 0.1352760, gradient norm = 0.0015652 (50 iterations in 0.015s)
[t-SNE] Iteration 350: error = 0.1247652, gradient norm = 0.0009720 (50 iterations in 0.017s)
[t-SNE] Iteration 400: error = 0.1214397, gradient norm = 

c:\Users\chq-anyac\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 250 samples in 0.000s...
[t-SNE] Computed neighbors for 250 samples in 0.009s...
[t-SNE] Computed conditional probabilities for sample 250 / 250
[t-SNE] Mean sigma: 0.651898
[t-SNE] Computed conditional probabilities in 0.007s
[t-SNE] Iteration 50: error = 57.7131958, gradient norm = 0.3160509 (50 iterations in 0.019s)
[t-SNE] Iteration 100: error = 58.0239944, gradient norm = 0.2802289 (50 iterations in 0.020s)
[t-SNE] Iteration 150: error = 59.0154800, gradient norm = 0.2883721 (50 iterations in 0.021s)
[t-SNE] Iteration 200: error = 58.5998306, gradient norm = 0.2847907 (50 iterations in 0.020s)
[t-SNE] Iteration 250: error = 58.0158463, gradient norm = 0.2905531 (50 iterations in 0.019s)
[t-SNE] KL divergence after 250 iterations with early exaggeration: 58.015846
[t-SNE] Iteration 300: error = 0.4734517, gradient norm = 0.0029524 (50 iterations in 0.019s)
[t-SNE] Iteration 350: error = 0.4484688, gradient norm = 0.0030171 (

View and interpret both the CLIP and ResNet-101 embeddings:  
i.e.  
- Looking at the results of t-SNE dimensionality reduction on both ResNet-101 and CLIP embeddings, we can see a lot more separation between the distributions of different classes.
- In both cases, similar classes are still close to each other — for instance, automobiles and trucks are adjacent — but we can also mostly distinguish a main cluster for almost every class. In other words, t-SNE does a very good job at capturing local structure, and a decent job at capturing global structure.

#### 1.1c Dimensionality Reduction with UMAP

[Uniform Manifold Approximation and Projection](https://umap-learn.readthedocs.io/en/latest/) (UMAP) is a nonlinear dimensionality reduction technique based on the mathematics of [topology](https://en.wikipedia.org/wiki/Topology). No gory details, as there is an excellent visual explanation of the approach [here](https://umap-learn.readthedocs.io/en/latest/how_umap_works.html), but in essence, UMAP treats the input data as points lying on a special kind of surface called a manifold (technically here a [Riemannian manifold](https://en.wikipedia.org/wiki/Riemannian_manifold)), and tries to learn a lower dimensional representation of the manifold. This explicitly takes global structure into consideration, as opposed to t-SNE, which concerns itself with keeping neighbors close (local structure).

**Strengths**

- Preserves both global and local structure
- Better scaling than t-SNE with dataset size

**Limitations**

- Like t-SNE, UMAP relies on randomness, and is dependent upon hyperparameters
- UMAP assumes that the manifold is locally connected. This can cause problems if there are a few data points that are very far away from the rest of the data.

#### 1.1c Running UMAP on Embeddings

UMAP  is natively supported by the FiftyOne Brain’s `compute_visualization()`, so we can run dimensionality reduction on our embeddings by passing `method="umap"`:

In [23]:
## UMAP with ResNet101 embeddings
fob.compute_visualization(
    dataset, 
    embeddings="resnet101_embeddings", 
    method="umap", 
    brain_key="resnet101_umap"
)

## UMAP with CLIP embeddings
fob.compute_visualization(
    dataset, 
    embeddings="clip_embeddings", 
    method="umap", 
    brain_key="clip_umap"
)

Generating visualization...


c:\Users\chq-anyac\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


UMAP( verbose=True)
Tue Jun  3 16:15:46 2025 Construct fuzzy simplicial set
Tue Jun  3 16:15:46 2025 Finding Nearest Neighbors
Tue Jun  3 16:15:51 2025 Finished Nearest Neighbor Search
Tue Jun  3 16:15:53 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Tue Jun  3 16:15:54 2025 Finished embedding
Generating visualization...
UMAP( verbose=True)
Tue Jun  3 16:15:54 2025 Construct fuzzy simplicial set
Tue Jun  3 16:15:54 2025 Finding Nearest Neighbors
Tue Jun  3 16:15:54 2025 Finished Nearest Neighbor Search
Tue Jun  3 16:15:54 2025 Construct embedding


c:\Users\chq-anyac\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Tue Jun  3 16:15:54 2025 Finished embedding


View and interpret both the CLIP and ResNet-101 embeddings:  
i.e.  
- For both sets of embeddings, the clusters are a lot more spread out than with t-SNE. For ResNet-101, all of the vehicles (automobile, truck, airplane, ship) are in one mega-cluster — or two smaller clusters, depending on how you view it — and all of the animals are in another mega-cluster.  
- Interestingly, for the CLIP embeddings, we see that the `airplane` cluster is situated close to both `bird` and `ship`. The `car` and `truck` clusters are very close together; and the `cat` and `dog` clusters are very close together.

In [24]:
#persist 51dataset
dataset.export(export_dir=EXPORT_PATH, dataset_type=fo.types.FiftyOneDataset) 

Exporting samples...
 100% |████████████████████| 250/250 [850.5ms elapsed, 0s remaining, 294.0 docs/s]      


## 2. Load 51Dataset from Disc
previously exported 

In [25]:
#Load persisted 51dataset from disc
loaded_dataset2 = fo.Dataset.from_dir(EXPORT_PATH, dataset_type=fo.types.FiftyOneDataset, name='doc-sample-invoices-250-exported') 
loaded_dataset2.__class__

Importing samples...
 100% |█████████████████| 250/250 [15.3ms elapsed, 0s remaining, 16.3K samples/s]      


fiftyone.core.dataset.Dataset

In [26]:
fo.list_datasets()

['clustering-demo',
 'clustering-demo2',
 'doc-images',
 'doc-images-2359',
 'doc-sample-invoices-250',
 'doc-sample-invoices-250-exported']

In [27]:
loaded_dataset2

Name:        doc-sample-invoices-250-exported
Media type:  image
Num samples: 250
Persistent:  False
Tags:        []
Sample fields:
    id:                   fiftyone.core.fields.ObjectIdField
    filepath:             fiftyone.core.fields.StringField
    tags:                 fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:             fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:           fiftyone.core.fields.DateTimeField
    last_modified_at:     fiftyone.core.fields.DateTimeField
    resnet101_embeddings: fiftyone.core.fields.VectorField
    clip_embeddings:      fiftyone.core.fields.VectorField

In [28]:
#launch if needed here
session = fo.launch_app(loaded_dataset2)


Could not connect session, trying again in 10 seconds



RuntimeError: Client is not connected

## Summary
Dimensionality reduction is critical to understanding our data, and our models. But it is important to think of dimensionality reduction not just as a single tool, but rather as a collection of techniques. Each technique has its own advantages; and each method projects certain assumptions onto the data, which may or may not hold for your data. I hope this walkthrough helps you to see your data in a new way!

In [ ]:
mystop

## Xtra

In [ ]:
# #$xtra 51 Sample code - add samples with gt_label
# import fiftyone as fo

# dataset = fo.load_dataset("name")

# samples = []
# for filepath, label in zip(filepaths, labels):
#     sample = fo.Sample(filepath=filepath)
#     sample["ground_truth"] = fo.Classification(label=label)
#     samples.append(sample)

# dataset.add_samples(samples)

In [ ]:
# $xtra 51 Sample code - add samples with gt_label and a field (one by one)
# dataset = fo.load_dataset("name")

# sample = fo.Sample(
#     filepath="DOC_IMAGES/DTW_2190599060_ANE_diGLmNgD3SlTBEw8wLOR8O0w2aCO53Spjdh_PAGE_0.jpg", 
#     ground_truth = fo.Classification(label="PackingList"))
# dataset.add_sample(sample)
# sample["carrier"] = "Carrier1"
# sample.save()


# sample2 = fo.Sample(
#     filepath="DOC_IMAGES/DTW_2190599388_BSH_WMDIcQgYdGnwXi2FCQOTj05G6LoKVy2xpaH_PAGE_0.jpg", 
#     ground_truth = fo.Classification(label="PurchaseOrder"))
# dataset.add_sample(sample2)
# sample2["carrier"] = "Carrier2"
# sample2.save()

In [ ]:
#$xtra Snippet delete ds document-images
# refer to https://docs.voxel51.com/user_guide/using_datasets.html
# dataset_docs.delete()
# fo.list_datasets()

In [ ]:
#$was: add samples with gt_label
#$fix need to set media type for the cell below to run, or produces an error
#$error: MediaTypeError: Unsupported media type 'None'
#$fix: refer to https://docs.voxel51.com/user_guide/using_datasets.html, looke for `media_type` property

#Reference
#task: Adding fields to a dataset  
# refer to https://docs.voxel51.com/user_guide/using_datasets.html#adding-fields-to-a-sample  
# https://docs.voxel51.com/user_guide/using_datasets.html#adding-fields-to-a-dataset

#dataset_docs.media_type #None

#add samples one by one
# sample = fo.Sample(filepath="DOC_IMAGES/DTW_2190599060_ANE_diGLmNgD3SlTBEw8wLOR8O0w2aCO53Spjdh_PAGE_0.jpg")
# dataset_docs.add_sample(sample)
# sample2 = fo.Sample(filepath="DOC_IMAGES/DTW_2190599388_BSH_WMDIcQgYdGnwXi2FCQOTj05G6LoKVy2xpaH_PAGE_0.jpg")
# dataset_docs.add_sample(sample2)

# #Load sample images and a gt_label
# if not DATASET_EXISTS:
#     i=0
#     for image in all_images:
#         #print(image)
#         dataset_docs.add_sample(fo.Sample(filepath=image, gt_label = fo.Classification(label=df_samples.iloc[i]['DOCUMENT_TYPE_NAME']))) #, tags=[df_samples.iloc[i]['DATA_SPLIT'], df_samples.iloc[i]['CARRIER']])))
#         i += 1

# #ensure right media type
# dataset_docs.media_type